# Etsy AI-Image Detector — Generalization Harness

Runs the **frozen** Etsy detector over generators it never trained on (GANs, diffusion, DALL·E, Midjourney, …) and reports **per-generator** metrics. The finding is *where the detector holds up and where it collapses* — the single 0.941 F1 hides this.

**Why this isn't a one-line `load_checkpoint`:** your detector is a multi-stage ensemble trained in-memory in `Etsy_final.ipynb`; the repo saves **no** artifacts. So Step 1 below serializes the trained pieces, then this notebook reloads them frozen and evaluates.

### Layered modes
- **`single`** — Intermediate-CLIP + XGBoost only (strongest single model, 0.923 val F1). One artifact, deterministic. **Run this first.**
- **`ensemble`** — full 4-model Optuna blend (0.941): CLIP + Inter-CLIP + EfficientNet(TTA) + FFT, with tuned weights/threshold.

### The gotchas that decide validity
1. Preprocessing matches training **by construction** (code copied from the training notebook). Don't add resizes.
2. Report **per generator**, never the average (this notebook refuses to).
3. Match real content to your training domain (Etsy-style product photos).
4. Watch the JPEG/PNG compression confound across `real/` vs fake folders.
5. Threshold may not transfer — trust AP/AUROC (threshold-free).

## 0. Install dependencies

In [ ]:
!pip install -q open-clip-torch xgboost scikit-learn joblib opencv-python pillow numpy tqdm huggingface_hub requests

## 1. Export artifacts from the training notebook (run ONCE)

The artifacts don't exist yet. **Paste the cell below at the END of `Etsy_final.ipynb`** — after every model is trained, so `xgb_inter`, `xgb_model`, `xgb_fft`, `fft_scaler`, `best_weights`, `best_threshold` and `best_efficientnet.pth` exist — and run it there. It writes the artifacts to Drive. (It is here for reference; it won't work in *this* notebook because the trained variables live in the training session.)

In [ ]:
# >>> PASTE THIS INTO Etsy_final.ipynb, NOT HERE <<<
import os, json, shutil, joblib

OUT = '/content/drive/MyDrive/etsy_artifacts'   # change if you like
os.makedirs(OUT, exist_ok=True)

# single-model artifact (always)
xgb_inter.save_model(os.path.join(OUT, 'xgb_inter.json'))
print('saved xgb_inter.json')

# ensemble artifacts (optional, for mode="ensemble")
try:
    xgb_model.save_model(os.path.join(OUT, 'xgb_clip.json'))   # CLIP final-layer XGB
    xgb_fft.save_model(os.path.join(OUT, 'xgb_fft.json'))
    joblib.dump(fft_scaler, os.path.join(OUT, 'fft_scaler.joblib'))
    shutil.copy('best_efficientnet.pth', os.path.join(OUT, 'best_efficientnet.pth'))
    ensemble = {'weights': best_weights, 'threshold': float(best_threshold)}
    with open(os.path.join(OUT, 'ensemble.json'), 'w') as f:
        json.dump(ensemble, f, indent=2)
    print('saved ensemble artifacts:', ensemble)
except NameError as e:
    print('ensemble artifacts skipped (missing var):', e, '- single mode still works.')

print('artifacts in', OUT, ':', sorted(os.listdir(OUT)))

## 2. Mount Drive & configure

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# ---- configuration ----
ARTIFACTS_DIR = '/content/drive/MyDrive/etsy_artifacts'  # from Step 1
DATA_DIR      = '/content/drive/MyDrive/data_eval'       # one folder per source (Step 4)
REAL_FOLDER   = 'real'      # name of the authentic-images subfolder under DATA_DIR
MODE          = 'single'    # 'single' or 'ensemble'
BATCH_SIZE    = 64
LIMIT         = 0           # cap images per folder (0 = all); set small for a smoke test
OUT_PREFIX    = 'reports/generalization'

## 3. Feature extractors

Copied faithfully from `Etsy_final.ipynb` so preprocessing at eval time is identical to training. Do not change the transforms or layer indices.

In [ ]:
"""
Feature extractors for the Etsy AI-image detector — copied faithfully from
notebooks/Etsy_final.ipynb so that preprocessing at evaluation time is IDENTICAL
to training. Do NOT "improve" these transforms; any change (extra resize, different
normalization, different layer indices) silently invalidates the generalization
numbers — see the preprocessing gotcha in generalization/README.md.

Covers the four signals used by the 0.941 ensemble:
  1. CLIP ViT-L/14 final embedding        (cell 3-5)
  2. Intermediate CLIP (resblocks 18/20/22/23 CLS tokens)  (cell 16-17)  <- strongest single model
  3. EfficientNet-B0 fine-tuned + TTA      (cell 6-10, 23)
  4. FFT radial power spectrum             (cell 18-19)
"""

from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms, models
from tqdm import tqdm

# --------------------------------------------------------------------------- #
# CLIP (final + intermediate)                                                  #
# --------------------------------------------------------------------------- #

_CLIP_CACHE = {}


def load_clip(device):
    """ViT-L-14 / laion2b_s32b_b82k — exactly as in notebook cell 3."""
    if "model" not in _CLIP_CACHE:
        import open_clip
        model, _, preprocess = open_clip.create_model_and_transforms(
            "ViT-L-14", pretrained="laion2b_s32b_b82k"
        )
        _CLIP_CACHE["model"] = model.to(device).eval()
        _CLIP_CACHE["preprocess"] = preprocess
    return _CLIP_CACHE["model"], _CLIP_CACHE["preprocess"]


@torch.no_grad()
def extract_clip_features(paths, device, batch_size=64):
    """Final CLIP embedding, L2-normalized. Mirrors notebook cell 4."""
    model, preprocess = load_clip(device)
    features = []
    for i in tqdm(range(0, len(paths), batch_size), desc="CLIP features"):
        batch_paths = paths[i:i + batch_size]
        images = []
        for p in batch_paths:
            try:
                images.append(preprocess(Image.open(p).convert("RGB")))
            except Exception:
                images.append(torch.zeros(3, 224, 224))
        batch = torch.stack(images).to(device)
        feats = model.encode_image(batch)
        feats = feats / feats.norm(dim=-1, keepdim=True)
        features.append(feats.cpu().numpy())
    return np.concatenate(features, axis=0)


@torch.no_grad()
def extract_intermediate_clip_features(paths, device,
                                       layer_indices=(18, 20, 22, 23),
                                       batch_size=64):
    """Concatenated CLS tokens from intermediate resblocks. Mirrors cell 16.

    This is the strongest single model (0.923 val F1) and the basis of the
    `single` evaluation mode.
    """
    model, preprocess = load_clip(device)
    layer_indices = list(layer_indices)
    activations = {}
    hooks = []
    transformer = model.visual.transformer

    for idx in layer_indices:
        layer = transformer.resblocks[idx]

        def hook_fn(module, inp, output, layer_idx=idx):
            out = output[0] if isinstance(output, tuple) else output
            if out.dim() == 3:
                # open_clip may emit (seq, batch, hidden) or (batch, seq, hidden)
                if out.shape[0] > out.shape[1]:
                    activations[layer_idx] = out[0]        # (batch, hidden)
                else:
                    activations[layer_idx] = out[:, 0]     # (batch, hidden)
            else:
                activations[layer_idx] = out

        hooks.append(layer.register_forward_hook(hook_fn))

    all_features = []
    try:
        for i in tqdm(range(0, len(paths), batch_size), desc="Intermediate CLIP"):
            batch_paths = paths[i:i + batch_size]
            images = []
            for p in batch_paths:
                try:
                    images.append(preprocess(Image.open(p).convert("RGB")))
                except Exception:
                    images.append(torch.zeros(3, 224, 224))
            batch = torch.stack(images).to(device)
            _ = model.encode_image(batch)
            feats = torch.cat([activations[l] for l in layer_indices], dim=-1)
            feats = feats / feats.norm(dim=-1, keepdim=True)
            all_features.append(feats.cpu().numpy())
    finally:
        for h in hooks:
            h.remove()
    return np.concatenate(all_features, axis=0)


# --------------------------------------------------------------------------- #
# EfficientNet-B0 (fine-tuned head) + TTA                                      #
# --------------------------------------------------------------------------- #

# val transform — notebook cell 6
val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# TTA transforms — notebook cell 23
tta_transforms = [
    val_transform,
    transforms.Compose([
        transforms.Resize((256, 256)), transforms.CenterCrop(224),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    transforms.Compose([
        transforms.Resize((256, 256)), transforms.CenterCrop(224),
        transforms.RandomRotation(degrees=(5, 5)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    transforms.Compose([
        transforms.Resize((256, 256)), transforms.CenterCrop(200),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    transforms.Compose([
        transforms.Resize((256, 256)), transforms.CenterCrop(224),
        transforms.GaussianBlur(kernel_size=3, sigma=0.5),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
]


def build_efficientnet(device):
    """Same architecture/head as notebook cell 7 (weights loaded separately)."""
    net = models.efficientnet_b0(weights=None)
    net.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(net.classifier[1].in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(256, 1),
    )
    return net.to(device).eval()


def load_efficientnet(checkpoint_path, device):
    net = build_efficientnet(device)
    state = torch.load(checkpoint_path, map_location=device, weights_only=True)
    net.load_state_dict(state)
    net.eval()
    return net


@torch.no_grad()
def efficientnet_probs(net, paths, device, transform=None, batch_size=64):
    """P(fake) for each path under a single transform."""
    transform = transform or val_transform
    probs = []
    for i in range(0, len(paths), batch_size):
        batch_paths = paths[i:i + batch_size]
        images = []
        for p in batch_paths:
            try:
                images.append(transform(Image.open(p).convert("RGB")))
            except Exception:
                images.append(torch.zeros(3, 224, 224))
        batch = torch.stack(images).to(device)
        logits = net(batch).squeeze(1)
        probs.extend(torch.sigmoid(logits).cpu().numpy())
    return np.array(probs)


@torch.no_grad()
def efficientnet_probs_tta(net, paths, device, batch_size=64):
    """TTA-averaged P(fake) — mirrors notebook cell 23."""
    stacks = [efficientnet_probs(net, paths, device, t, batch_size)
              for t in tta_transforms]
    return np.mean(stacks, axis=0)


# --------------------------------------------------------------------------- #
# FFT radial power spectrum                                                    #
# --------------------------------------------------------------------------- #

def extract_fft_features(image_path, img_size=256):
    """Radial power spectrum + hf_ratio + spectral_centroid. Mirrors cell 18."""
    import cv2
    try:
        img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (img_size, img_size))
    except Exception:
        img = np.zeros((img_size, img_size), dtype=np.uint8)
    f_transform = np.fft.fft2(img.astype(np.float32))
    f_shift = np.fft.fftshift(f_transform)
    magnitude = np.log1p(np.abs(f_shift))
    cy, cx = img_size // 2, img_size // 2
    y_coords, x_coords = np.ogrid[:img_size, :img_size]
    r = np.sqrt((x_coords - cx) ** 2 + (y_coords - cy) ** 2).astype(int)
    max_r = min(cx, cy)
    power_spectrum = np.zeros(max_r)
    for radius in range(max_r):
        mask = r == radius
        if mask.any():
            power_spectrum[radius] = magnitude[mask].mean()
    total_energy = power_spectrum.sum()
    if total_energy > 0:
        hf_ratio = power_spectrum[max_r // 2:].sum() / total_energy
        spectral_centroid = np.sum(np.arange(max_r) * power_spectrum) / total_energy
    else:
        hf_ratio, spectral_centroid = 0.0, 0.0
    return np.concatenate([power_spectrum, [hf_ratio, spectral_centroid]])


def build_fft_matrix(paths):
    feats = [extract_fft_features(p) for p in tqdm(paths, desc="FFT features")]
    return np.array(feats)

## 4. Detector (loads frozen artifacts) & metrics

In [ ]:
"""
EtsyDetector — frozen-inference wrapper around the trained artifacts.

Two layered modes (see README):
  mode="single"   -> Intermediate-CLIP + xgb_inter only (0.923 model).
                     Needs ONE artifact: xgb_inter.json. Deterministic, no GPU
                     dependence beyond CLIP, no Optuna, no EfficientNet.
  mode="ensemble" -> Full 4-model Optuna blend (0.941): CLIP + Inter-CLIP +
                     EfficientNet(TTA) + FFT, weighted by ensemble.json.

Artifacts are produced by export_artifacts_cell.py, run at the end of the
training notebook. Nothing here retrains anything.
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import torch
from xgboost import XGBClassifier

# extractors are defined inline above; alias module namespace
import sys as _sys
ex = _sys.modules[__name__]


def _load_xgb(path):
    clf = XGBClassifier()
    clf.load_model(str(path))
    return clf


class EtsyDetector:
    def __init__(self, artifacts_dir, mode="single", device=None):
        self.dir = Path(artifacts_dir)
        self.mode = mode
        self.device = torch.device(
            device or ("cuda" if torch.cuda.is_available() else "cpu")
        )

        # always needed
        self.xgb_inter = _load_xgb(self.dir / "xgb_inter.json")

        if mode == "ensemble":
            self.xgb_clip = _load_xgb(self.dir / "xgb_clip.json")
            self.xgb_fft = _load_xgb(self.dir / "xgb_fft.json")
            import joblib
            self.fft_scaler = joblib.load(self.dir / "fft_scaler.joblib")
            self.effnet = ex.load_efficientnet(
                self.dir / "best_efficientnet.pth", self.device
            )
            with open(self.dir / "ensemble.json") as f:
                ens = json.load(f)
            # weights keyed clip/inter/eff/fft; threshold tuned on val
            self.weights = ens["weights"]
            self.threshold = ens["threshold"]
        else:
            self.weights = None
            self.threshold = 0.5

    # ------------------------------------------------------------------ #
    def score_paths(self, paths, batch_size=64):
        """Return P(fake) array aligned with `paths`."""
        paths = [str(p) for p in paths]

        inter = ex.extract_intermediate_clip_features(
            paths, self.device, batch_size=batch_size
        )
        inter_p = self.xgb_inter.predict_proba(inter)[:, 1]
        if self.mode == "single":
            return inter_p

        clip = ex.extract_clip_features(paths, self.device, batch_size=batch_size)
        clip_p = self.xgb_clip.predict_proba(clip)[:, 1]

        eff_p = ex.efficientnet_probs_tta(
            self.effnet, paths, self.device, batch_size=batch_size
        )

        fft = ex.build_fft_matrix(paths)
        fft_p = self.xgb_fft.predict_proba(self.fft_scaler.transform(fft))[:, 1]

        w = self.weights
        return (w["clip"] * clip_p + w["inter"] * inter_p
                + w["eff"] * eff_p + w["fft"] * fft_p)

    def decision_threshold(self):
        """0.5 for single mode; Optuna-tuned threshold for ensemble."""
        return self.threshold

In [ ]:
"""
Per-generator metrics — the actual finding.

Each fake generator is paired against the SAME real set, so every row isolates
"can the frozen detector spot fakes from THIS generator." We report:
  ACC@0.5            accuracy at the fixed 0.5 threshold
  ACC@thr           accuracy at the model's own decision threshold (ensemble)
  AP, AUROC         threshold-free separability (the honest numbers)
  F1                F1 at the model's decision threshold
Never collapse this to one average — the aggregate hides the collapse you are
trying to surface (a detector can be 0.94 overall and ~0.5 on DALL-E 3).
"""

from __future__ import annotations

import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score


def per_generator_metrics(real_probs, fake_probs_by_gen, threshold=0.5):
    """
    real_probs        : np.array of P(fake) for real images (label 0)
    fake_probs_by_gen : dict[str, np.array] of P(fake) per generator (label 1)
    threshold         : model decision threshold for ACC@thr / F1
    Returns a list of metric dicts, one row per generator.
    """
    rows = []
    real_y = np.zeros(len(real_probs))
    for gen, fake_p in fake_probs_by_gen.items():
        fake_y = np.ones(len(fake_p))
        y = np.concatenate([real_y, fake_y])
        p = np.concatenate([real_probs, fake_p])

        row = {
            "generator": gen,
            "n_fake": int(len(fake_p)),
            "n_real": int(len(real_probs)),
            "ACC@0.5": round(float(((p > 0.5).astype(int) == y).mean()), 4),
        }
        # only add a separate threshold column when it differs from 0.5
        if abs(threshold - 0.5) > 1e-9:
            row[f"ACC@{threshold:.2f}"] = round(
                float(((p >= threshold).astype(int) == y).mean()), 4)
        row.update({
            "AP": round(float(average_precision_score(y, p)), 4),
            "AUROC": round(float(roc_auc_score(y, p)), 4),
            "F1": round(float(f1_score(y, (p >= threshold).astype(int))), 4),
            "recall_fake@thr": round(float((fake_p >= threshold).mean()), 4),
        })
        rows.append(row)
    return rows


def format_table(rows):
    """Plain-text table for stdout/report."""
    if not rows:
        return "(no results)"
    cols = list(rows[0].keys())
    widths = {c: max(len(c), *(len(str(r[c])) for r in rows)) for c in cols}
    line = "  ".join(c.ljust(widths[c]) for c in cols)
    out = [line, "  ".join("-" * widths[c] for c in cols)]
    for r in rows:
        out.append("  ".join(str(r[c]).ljust(widths[c]) for c in cols))
    return "\n".join(out)


def to_markdown(rows):
    if not rows:
        return "(no results)"
    cols = list(rows[0].keys())
    head = "| " + " | ".join(cols) + " |"
    sep = "| " + " | ".join("---" for _ in cols) + " |"
    body = ["| " + " | ".join(str(r[c]) for c in cols) + " |" for r in rows]
    return "\n".join([head, sep] + body)

## 5. (Optional) Download & lay out datasets

The harness needs `DATA_DIR` with one subfolder per source: `real/` plus one folder per generator. Helpers below fetch the public benchmarks and reshape them. Repo IDs / Zenodo records drift — verify against the canonical pages.

- **GenImage** (Midjourney, SD1.4/1.5, Wukong, VQDM, ADM, GLIDE, BigGAN; real = ImageNet) — arxiv.org/abs/2306.08571; mirrors on HF Hub.
- **Synthbuster** (DALL·E 2/3, Firefly, Midjourney v5, SD1.3/1.4/2/XL, GLIDE; real = RAISE-1k) — arxiv.org/abs/2304.06184; Zenodo.
- **ForenSynths/UFD** — pure-GAN suite.

In [ ]:
"""
Dataset download + layout helpers.

The harness only cares about ONE contract: a directory with one subfolder per
source, `real/` plus one folder per generator. These helpers fetch the public
benchmarks and reshape them into that contract. Repo IDs / Zenodo records are
left as arguments because they drift — verify against the canonical pages below.

Canonical sources
-----------------
GenImage   (~1M imgs; Midjourney, SD1.4/1.5, Wukong, VQDM, ADM, GLIDE, BigGAN;
            real = ImageNet). Paper: arxiv.org/abs/2306.08571
            Mirrors live on Hugging Face Hub (search "GenImage"); also a Google
            Drive release. Per-generator folders already split ai/ vs nature/.
Synthbuster (9k imgs; DALL-E 2/3, Firefly, Midjourney v5, SD1.3/1.4/2/XL, GLIDE;
            real = RAISE-1k). Paper: arxiv.org/abs/2304.06184
            Zenodo: search "Synthbuster" (record id changes between versions).
ForenSynths/UFD  (pure-GAN suite: ProGAN, StyleGAN(2), BigGAN, CycleGAN,
            StarGAN, GauGAN). Used by UniversalFakeDetect / RINE main table.

NOTE: these are large. Start with --limit on the harness for a smoke test, and
download a single generator subset first.
"""

from __future__ import annotations

import shutil
from pathlib import Path

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


def download_hf(repo_id, local_dir, repo_type="dataset", allow_patterns=None):
    """Snapshot a Hugging Face dataset repo (e.g. a GenImage mirror)."""
    from huggingface_hub import snapshot_download
    path = snapshot_download(
        repo_id=repo_id,
        repo_type=repo_type,
        local_dir=local_dir,
        allow_patterns=allow_patterns,   # e.g. ["stable_diffusion_v_1_5/*"]
    )
    print("downloaded to", path)
    return path


def download_zenodo(record_id, local_dir):
    """Fetch all files from a Zenodo record (e.g. Synthbuster)."""
    import requests
    local = Path(local_dir)
    local.mkdir(parents=True, exist_ok=True)
    meta = requests.get(f"https://zenodo.org/api/records/{record_id}").json()
    files = meta.get("files", [])
    if not files:
        raise SystemExit(f"No files on Zenodo record {record_id}: {meta.get('message')}")
    for f in files:
        url = f["links"]["self"]
        dest = local / f["key"]
        print("downloading", f["key"], f.get("size"))
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            with open(dest, "wb") as out:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    out.write(chunk)
    print("downloaded", len(files), "files to", local)
    return str(local)


def link_source(src_dir, out_dir, name, limit=0, copy=False):
    """
    Expose an arbitrary folder of images as out_dir/<name>/ for the harness.
    By default symlinks the folder (cheap); pass copy=True to copy a sample.

    Example, building an eval set from a GenImage download:
        link_source(".../stable_diffusion_v_1_5/ai",     "data_eval", "sd15")
        link_source(".../stable_diffusion_v_1_5/nature", "data_eval", "real")
    """
    out = Path(out_dir) / name
    src = Path(src_dir)
    if copy or limit:
        out.mkdir(parents=True, exist_ok=True)
        imgs = sorted(p for p in src.rglob("*") if p.suffix.lower() in IMG_EXTS)
        if limit:
            imgs = imgs[:limit]
        for p in imgs:
            shutil.copy(p, out / p.name)
        print(f"copied {len(imgs)} -> {out}")
    else:
        out.parent.mkdir(parents=True, exist_ok=True)
        if out.exists() or out.is_symlink():
            out.unlink()
        out.symlink_to(src.resolve(), target_is_directory=True)
        print(f"linked {src} -> {out}")
    return str(out)

In [ ]:
# Example: reshape a downloaded GenImage subset into the DATA_DIR contract.
# (uncomment & edit paths)
#
# p = download_hf('SOME/genimage-mirror', '/content/genimage',
#                 allow_patterns=['stable_diffusion_v_1_5/*'])
# link_source(f'{p}/stable_diffusion_v_1_5/ai',     DATA_DIR, 'sd15')
# link_source(f'{p}/stable_diffusion_v_1_5/nature', DATA_DIR, REAL_FOLDER)
#
# Synthbuster (DALL-E / Midjourney):
# d = download_zenodo(RECORD_ID, '/content/synthbuster')
# link_source(f'{d}/dalle3',     DATA_DIR, 'dalle3')
# link_source(f'{d}/midjourney', DATA_DIR, 'midjourney')

## 6. Run the harness

Scores `real/` once, scores every generator folder, computes per-generator metrics against that shared real set, prints the table (worst-AUROC first), and writes CSV + Markdown + JSON.

In [ ]:
import csv, json
from pathlib import Path

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}

def list_images(folder, limit=0):
    imgs = sorted(p for p in Path(folder).rglob('*') if p.suffix.lower() in IMG_EXTS)
    return imgs[:limit] if limit else imgs

data = Path(DATA_DIR)
real_dir = data / REAL_FOLDER
assert real_dir.is_dir(), f'Real folder not found: {real_dir}'
gen_dirs = sorted(d for d in data.iterdir() if d.is_dir() and d.name != REAL_FOLDER)
assert gen_dirs, f'No generator folders under {data}'

det = EtsyDetector(ARTIFACTS_DIR, mode=MODE)
thr = det.decision_threshold()
print(f'Mode: {MODE} | device: {det.device} | threshold: {thr:.3f}')

def score(folder):
    paths = list_images(folder, LIMIT)
    if not paths:
        return None
    print(f'  scoring {len(paths):>6d} images in {folder.name}/')
    return det.score_paths(paths, batch_size=BATCH_SIZE)

print('Real set:', real_dir)
real_probs = score(real_dir)
assert real_probs is not None, 'No images in real folder.'

fake_by_gen = {}
for d in gen_dirs:
    p = score(d)
    if p is not None:
        fake_by_gen[d.name] = p

rows = per_generator_metrics(real_probs, fake_by_gen, threshold=thr)
rows.sort(key=lambda r: r['AUROC'])   # worst-generalizing first = the finding

print('\n=== Per-generator generalization ===')
print(format_table(rows))

## 7. Save results

In [ ]:
out = Path(OUT_PREFIX)
out.parent.mkdir(parents=True, exist_ok=True)

with open(f'{out}_{MODE}.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)
with open(f'{out}_{MODE}.md', 'w') as f:
    f.write(f'# Generalization — {MODE} model\n\n')
    f.write(f'Real set: `{REAL_FOLDER}` ({rows[0]["n_real"]} images) · threshold {thr:.3f}\n\n')
    f.write(to_markdown(rows) + '\n')
with open(f'{out}_{MODE}.json', 'w') as f:
    json.dump({'mode': MODE, 'threshold': thr, 'rows': rows}, f, indent=2)

print(f'Wrote {out}_{MODE}.csv / .md / .json')

# render the table inline
try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    pass